# GPT-5.4-mini(API) Vision을 활용한 정치인 이미지 분석

참고 문서: [Latest Model Guide](https://developers.openai.com/api/docs/guides/latest-model)

---

## 실행 환경 요약

- 모델: `gpt-5.4-mini` / API: OpenAI **Responses API** (`client.responses.create`) + JSON Schema 구조화 출력(`strict: true`)
- `reasoning_effort="none"` (분류·추출·요약 과제에 적합). `temperature`는 `reasoning_effort="none"`일 때만 지정할 수 있습니다.
- 이전 버전: [GPT-4.1 버전 저장소](https://github.com/HanJunLeeKR/News_Content_Analysis_with_GPT-4.1) (Chat Completions + `json_object`)
- 두 버전의 차이는 저장소 `README.md`의 「GPT-4.1 버전과의 차이」를 참고하세요.
- 이미지 `detail` 옵션: `low` / `high` / `original`(이미지 내 텍스트·슬로건 OCR이 필요할 때)

---

## 적용 코딩 스킴 (3차원)

| 차원 | 카테고리 수 | 이론적 근거 |
|---|---|---|
| **이미지 내용 유형** | 5개 | Grabe & Bucy (2009) |
| **감정적 어조** | 4개 | Brader (2006) |
| **후보자 표현 방식** | 4개 | Schill (2012) |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 패키지 설치 (Colab/Jupyter용). 로컬 환경에서는 저장소의 requirements.txt 사용:
#   pip install -r requirements.txt
# ═══════════════════════════════════════════════════════════════════════
%pip install --upgrade openai pandas tqdm nest-asyncio


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. 임포트
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Tuple, Dict, Optional
import re
from tqdm import tqdm


# ═══════════════════════════════════════════════════════════════════════
# 2. 정치 이미지 코딩 스킴 정의 
# ═══════════════════════════════════════════════════════════════════════
CODING_SCHEME = {
    "image_content_type": {
        "인물 초상 (Portrait)": (
            "정치인 단독 또는 소수 인원의 공식·비공식 촬영. "
            "인물이 화면 전면에 배치되며 배경이 단순하거나 없음."
        ),
        "유세/집회 (Campaign Rally)": (
            "군중, 지지자, 또는 대규모 행사 장면. "
            "정치인이 청중 앞에서 연설하거나 군중 속에 있음."
        ),
        "정책/현장 (Policy/Field Visit)": (
            "정책 발표, 현장 방문, 기관 시찰, 협약식 등 "
            "구체적 정책 활동과 연관된 장면."
        ),
        "공격/대립 (Attack/Opposition)": (
            "상대 후보나 정당을 비판하거나, 비리·실패 등 "
            "부정적 이미지를 전면에 내세우는 내용."
        ),
        "일상/친근 (Humanizing/Everyday)": (
            "정치인의 일상, 가족과의 모습, 취미, 비공식 활동 등 "
            "인간적 면모를 강조하는 장면."
        ),
    },
    "emotional_tone": {
        "희망/긍정 (Hope/Positive)": (
            "밝은 미래, 낙관, 긍정적 변화를 강조하는 시각적 정서. "
            "밝은 조명, 미소, 활기찬 색채가 특징."
        ),
        "위협/공포 (Threat/Fear)": (
            "현재 상황의 위험성, 위기감, 불안을 유발하는 시각적 정서. "
            "어두운 톤, 긴박한 표정, 경고성 이미지가 특징."
        ),
        "분노/갈등 (Anger/Conflict)": (
            "불의나 잘못에 대한 분노, 대립, 저항을 담은 시각적 정서. "
            "강한 제스처, 대립 구도, 긴장감이 특징."
        ),
        "중립/정보 (Neutral/Informational)": (
            "특정 감정보다 사실 전달에 집중하는 중립적 정서. "
            "감정적 과장 없이 상황을 있는 그대로 보여줌."
        ),
    },
    "candidate_presentation": {
        "권위/전문성 (Authority/Competence)": (
            "공식 복장, 진지한 표정, 전문적 환경을 통해 "
            "지도자로서의 능력과 신뢰를 강조."
        ),
        "친근성/공감 (Warmth/Empathy)": (
            "미소, 악수, 시민과의 신체 접촉 등을 통해 "
            "인간적 공감대와 친근함을 강조."
        ),
        "행동력/역동성 (Action/Dynamism)": (
            "연설, 강한 제스처, 움직임을 통해 "
            "결단력, 에너지, 리더십을 강조."
        ),
        "피해자/도전자 (Victim/Challenger)": (
            "불공정이나 기득권에 맞서는 약자 또는 도전자로서의 "
            "이미지를 강조. 억울함·저항 포지셔닝."
        ),
    },
}


def format_coding_scheme() -> str:
    """코딩 스킴을 프롬프트용 문자열로 변환"""
    labels = {
        "image_content_type":     "[차원 1] 이미지 내용 유형 (Image Content Type)",
        "emotional_tone":         "[차원 2] 감정적 어조 (Emotional Tone)",
        "candidate_presentation": "[차원 3] 후보자 표현 방식 (Candidate Presentation)",
    }
    sections = []
    for key, categories in CODING_SCHEME.items():
        section = f"{labels[key]}\n"
        for cat_name, cat_desc in categories.items():
            section += f"  • {cat_name}: {cat_desc}\n"
        sections.append(section)
    return "\n".join(sections)


# ═══════════════════════════════════════════════════════════════════════
# 3. 프롬프트 템플릿
#    GPT-5.4 prompt guidance 반영:
#    - Output contract 명시 (카테고리명 정확 사용)
#    - 검증 루프 명시 (JSON 형식 확인 지시)
#    - 더 리터럴한(literal) 지시 방식 채택
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """You are an expert researcher in political visual communication.
Analyze the provided political image and classify it across exactly 3 dimensions.

{coding_scheme}

CLASSIFICATION RULES:
1. Examine the full visual composition: subject placement, lighting, color palette, facial expression, body language, and background context.
2. For each dimension, select EXACTLY ONE category from the options listed above.
3. The value of each field MUST be copied verbatim from the category names above (e.g., '인물 초상 (Portrait)').
4. In the `reason` field, provide at least 2 sentences citing specific visual elements that justify all three classifications.
5. Set `confidence` as an integer from 1 to 100 reflecting your certainty across all three dimensions.
6. Output ONLY valid JSON. Do not add fields not in the schema."""

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. JSON 파서 
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""

    REQUIRED_KEYS = ["image_content_type", "emotional_tone", "candidate_presentation"]

    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        if not response_text or not response_text.strip():
            return EnhancedJSONParser._get_default_result("빈 응답")

        response_text = response_text.strip()

        # 1단계: 직접 파싱
        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                parsed = json.loads(response_text)
                if EnhancedJSONParser._validate_result(parsed):
                    return parsed
        except json.JSONDecodeError:
            pass

        # 2단계: 마크다운 블록
        for pattern in [r'```json\s*\n?(.*?)\n?```', r'```\s*\n?(.*?)\n?```']:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                try:
                    parsed = json.loads(match.group(1).strip())
                    if EnhancedJSONParser._validate_result(parsed):
                        return parsed
                except json.JSONDecodeError:
                    continue

        # 3단계: 중괄호 블록
        for match in re.findall(r'\{(?:[^{}]|\{[^{}]*\})*\}', response_text):
            try:
                parsed = json.loads(match)
                if EnhancedJSONParser._validate_result(parsed):
                    return parsed
            except json.JSONDecodeError:
                continue

        return EnhancedJSONParser._get_default_result("파싱 실패")

    @staticmethod
    def _validate_result(result: Dict) -> bool:
        return (
            isinstance(result, dict)
            and all(k in result for k in EnhancedJSONParser.REQUIRED_KEYS)
            and all(isinstance(result[k], str) and result[k].strip() for k in EnhancedJSONParser.REQUIRED_KEYS)
        )

    @staticmethod
    def _get_default_result(error_msg: str) -> Dict:
        return {
            "image_content_type":     "분류 실패",
            "emotional_tone":         "분류 실패",
            "candidate_presentation": "분류 실패",
            "reason":     f"파싱 오류: {error_msg}",
            "confidence": 0,
        }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 5. 정치 이미지 분석기 (GPT-5.4-mini Vision / Responses API)
#
# [GPT-5.4-mini 주요 설정]
#
#  ① reasoning.effort: "none" 사용
#     옵션: none | low | medium | high | xhigh
#     분류 작업은 "none" 권장 (빠른 추출·변환 태스크)
#
#  ② temperature: reasoning="none" 일 때만 사용 가능
#     다른 reasoning level에서 temperature 포함 시 API 오류 발생!
#
#  ③ image detail: "original" 옵션 추가
#     high    → 표준 이미지 분석 (권장)
#     original → 이미지 내 텍스트·배너·슬로건 OCR 필요 시
#     low      → 속도 우선 (정밀도 낮음)
#
#  ④ phase 파라미터: 선택적 신규 기능
#     단일 턴 분류에는 불필요 → 이 코드에서는 미사용
# ═══════════════════════════════════════════════════════════════════════
class PoliticalImageAnalyzer:
    """GPT-5.4-mini Vision을 활용한 정치인 이미지 분석기"""

    def __init__(
        self,
        api_key: str,
        model: str,
        output_dir: str = ".",
        concurrent_requests: int = 2,
        rate_limit_delay: float = 1.0,
        batch_size: int = 5,
        reasoning_effort: str = "none",       # ★ GPT-5.4: "none" 권장 (분류 태스크)
        text_verbosity: str = "medium",
        image_detail: str = "high",            # ★ GPT-5.4: "original" 추가됨 (OCR용)
        temperature: Optional[float] = None,   # ★ GPT-5.4: reasoning="none" 일 때만 유효
        debug: bool = False,
    ):
        self.client              = AsyncOpenAI(api_key=api_key)
        self.model               = model
        self.output_dir          = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay    = rate_limit_delay
        self.batch_size          = batch_size
        self.reasoning_effort    = reasoning_effort
        self.text_verbosity      = text_verbosity
        self.image_detail        = image_detail
        self.temperature         = temperature
        self.debug               = debug
        self.total_cost          = 0.0
        self.semaphore           = asyncio.Semaphore(concurrent_requests)
        self.parsing_successes   = 0
        self.parsing_failures    = 0

        # temperature는 reasoning=none 일 때만 허용
        if self.temperature is not None and self.reasoning_effort != "none":
            raise ValueError(
                f"[GPT-5.4 제약] temperature는 reasoning_effort='none' 일 때만 사용 가능합니다. "
                f"현재 설정: reasoning_effort='{self.reasoning_effort}'"
            )

        logging.basicConfig(level=logging.WARNING, format="%(message)s")
        self.logger = logging.getLogger(__name__)
        os.makedirs(self.output_dir, exist_ok=True)

    # ───────────────────────────────────────────────────────────────────
    # 단일 이미지 분석
    # ───────────────────────────────────────────────────────────────────
    async def analyze_single_image(
        self, image_url: str, row_id: str = ""
    ) -> Tuple[str, str, str, str, int, float]:
        """단일 이미지 URL 분석 → (이미지유형, 감정톤, 후보표현방식, 근거, 신뢰도, 비용)"""

        async with self.semaphore:
            try:
                if not image_url or pd.isna(image_url) or str(image_url).strip() == "":
                    return "분류 실패", "분류 실패", "분류 실패", "빈 URL", 0, 0.0

                image_url = str(image_url).strip()
                prompt    = PROMPT_TEMPLATE.format(coding_scheme=format_coding_scheme())

                # ─────────────────────────────────────────────────────
                # Responses API 파라미터 구성
                # ─────────────────────────────────────────────────────
                api_params = dict(
                    model=self.model,
                    input=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    # ★ GPT-5.4: detail="original" → 이미지 내 텍스트 정밀 분석
                                    "type":      "input_image",
                                    "image_url": image_url,
                                    "detail":    self.image_detail,
                                },
                                {
                                    "type": "input_text",
                                    "text": prompt,
                                },
                            ],
                        }
                    ],
                    max_output_tokens=1024,
                    reasoning={"effort": self.reasoning_effort},   # ★ "none"|"low"|"medium"|"high"|"xhigh"
                    text={
                        "verbosity": self.text_verbosity,
                        "format": {
                            "type":   "json_schema",
                            "name":   "political_image_analysis",
                            "strict": True,
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "image_content_type": {
                                        "type":        "string",
                                        "description": "이미지 내용 유형 — 5개 카테고리 중 1개, 카테고리명 그대로",
                                    },
                                    "emotional_tone": {
                                        "type":        "string",
                                        "description": "감정적 어조 — 4개 카테고리 중 1개, 카테고리명 그대로",
                                    },
                                    "candidate_presentation": {
                                        "type":        "string",
                                        "description": "후보자 표현 방식 — 4개 카테고리 중 1개, 카테고리명 그대로",
                                    },
                                    "reason": {
                                        "type":        "string",
                                        "description": "3차원 분류 근거 (시각적 요소 기반, 2문장 이상)",
                                    },
                                    "confidence": {
                                        "type":        "integer",
                                        "description": "분류 종합 신뢰도 (1-100)",
                                        "minimum":     1,
                                        "maximum":     100,
                                    },
                                },
                                "required": [
                                    "image_content_type",
                                    "emotional_tone",
                                    "candidate_presentation",
                                    "reason",
                                    "confidence",
                                ],
                                "additionalProperties": False,
                            },
                        },
                    },
                )

                # ★ GPT-5.4: temperature는 reasoning="none" 일 때만 추가
                if self.temperature is not None and self.reasoning_effort == "none":
                    api_params["temperature"] = self.temperature

                response = await self.client.responses.create(**api_params)

                # ─────────────────────────────────────────────────────
                # 응답 텍스트 추출
                # ─────────────────────────────────────────────────────
                if hasattr(response, "output_text") and response.output_text:
                    response_text = response.output_text
                else:
                    response_text = ""
                    for item in response.output:
                        if item.type == "message":
                            for content in item.content:
                                if hasattr(content, "text"):
                                    response_text += content.text

                response_text = response_text.strip()
                if self.debug:
                    print(f"[{row_id}] raw: {response_text[:200]}")

                if not response_text:
                    self.parsing_failures += 1
                    return "분류 실패", "분류 실패", "분류 실패", "빈 응답", 0, 0.0

                # JSON 파싱
                result = EnhancedJSONParser.extract_json_robust(response_text)

                img_type  = result.get("image_content_type",    "분류 실패")
                emo_tone  = result.get("emotional_tone",        "분류 실패")
                cand_pres = result.get("candidate_presentation", "분류 실패")
                reason    = result.get("reason",     "파싱 실패")
                confidence = result.get("confidence", 0)

                try:
                    confidence = max(0, min(100, int(confidence)))
                except (ValueError, TypeError):
                    confidence = 0

                if img_type != "분류 실패":
                    self.parsing_successes += 1
                else:
                    self.parsing_failures += 1

                # 비용 계산 (GPT-5.4-mini 예상가 — 실제 가격은 OpenAI Pricing 페이지 확인)
                input_tokens  = getattr(response.usage, "input_tokens",  0)
                output_tokens = getattr(response.usage, "output_tokens", 0)
                estimated_cost = (
                    (input_tokens  * PRICE_INPUT_PER_1M / 1_000_000)
                    + (output_tokens * PRICE_OUTPUT_PER_1M / 1_000_000)
                )
                self.total_cost += estimated_cost

                if self.debug:
                    print(f"[{row_id}] {img_type} | {emo_tone} | {cand_pres} (conf={confidence})")

                return img_type, emo_tone, cand_pres, reason, confidence, estimated_cost

            except Exception as e:
                if self.debug:
                    print(f"[DEBUG] {type(e).__name__}: {str(e)}")
                self.parsing_failures += 1
                return "분류 실패", "분류 실패", "분류 실패", f"오류: {str(e)[:80]}", 0, 0.0

            finally:
                await asyncio.sleep(self.rate_limit_delay)

    # ───────────────────────────────────────────────────────────────────
    # 배치 처리
    # ───────────────────────────────────────────────────────────────────
    async def process_batch(self, batch_df) -> Dict:
        tasks = [
            self.analyze_single_image(row["image"], str(idx))
            for idx, row in batch_df.iterrows()
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        out = {k: [] for k in ["types", "tones", "presentations", "reasons", "confidences", "costs"]}
        for r in results:
            if isinstance(r, Exception):
                out["types"].append("분류 실패")
                out["tones"].append("분류 실패")
                out["presentations"].append("분류 실패")
                out["reasons"].append(f"배치 오류: {str(r)[:50]}")
                out["confidences"].append(0)
                out["costs"].append(0.0)
                self.parsing_failures += 1
            else:
                out["types"].append(r[0])
                out["tones"].append(r[1])
                out["presentations"].append(r[2])
                out["reasons"].append(r[3])
                out["confidences"].append(r[4])
                out["costs"].append(r[5])
        return out

    # ───────────────────────────────────────────────────────────────────
    # 전체 DataFrame 분석
    # ───────────────────────────────────────────────────────────────────
    async def analyze_dataframe(
        self, df: pd.DataFrame, output_filename: str = "political_image_results.csv"
    ) -> pd.DataFrame:

        output_path = os.path.join(self.output_dir, output_filename)
        all_types, all_tones, all_presentations = [], [], []
        all_reasons, all_confidences, all_costs  = [], [], []
        start_time = time.time()

        pbar = tqdm(total=len(df), desc="정치 이미지 분석 중", unit="건")

        for i in range(0, len(df), self.batch_size):
            batch_df = df.iloc[i : min(i + self.batch_size, len(df))]
            try:
                out = await self.process_batch(batch_df)
                all_types.extend(out["types"])
                all_tones.extend(out["tones"])
                all_presentations.extend(out["presentations"])
                all_reasons.extend(out["reasons"])
                all_confidences.extend(out["confidences"])
                all_costs.extend(out["costs"])
                pbar.update(len(batch_df))

                if (i // self.batch_size + 1) % 5 == 0:
                    self._save_intermediate(
                        df, all_types, all_tones, all_presentations,
                        all_reasons, all_confidences, all_costs, output_path
                    )
            except Exception as e:
                n = len(batch_df)
                all_types.extend(["분류 실패"] * n)
                all_tones.extend(["분류 실패"] * n)
                all_presentations.extend(["분류 실패"] * n)
                all_reasons.extend([f"배치 오류: {str(e)[:30]}"] * n)
                all_confidences.extend([0] * n)
                all_costs.extend([0.0] * n)
                self.parsing_failures += n
                pbar.update(n)

        pbar.close()

        result_df = df.copy()
        result_df["이미지유형"]   = all_types
        result_df["감정톤"]       = all_tones
        result_df["후보표현방식"] = all_presentations
        result_df["근거"]         = all_reasons
        result_df["신뢰도"]       = all_confidences
        result_df["비용"]         = all_costs
        result_df.to_csv(output_path, encoding="utf-8-sig", index=False)

        self._print_completion_message(result_df, time.time() - start_time, output_path)
        return result_df

    def _save_intermediate(self, df, types, tones, presentations, reasons, confidences, costs, path):
        n = len(df)
        temp = df.copy()
        temp["이미지유형"]   = types         + [""]
        temp["감정톤"]       = tones         + [""]
        temp["후보표현방식"] = presentations + [""]
        temp["근거"]         = reasons       + [""]
        temp["신뢰도"]       = confidences   + [0]
        temp["비용"]         = costs         + [0.0]
        # 길이 맞추기
        for col in ["이미지유형", "감정톄", "후보표현방식", "근거"]:
            if col in temp.columns:
                temp[col] = temp[col][:n]
        temp.to_csv(path, encoding="utf-8-sig", index=False)

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        total = len(result_df)
        print(f"\n{'='*62}")
        print(f"✅ 정치 이미지 분석 완료! (GPT-5.4-mini Vision)")
        print(f"{'='*62}")
        print(f"📊 총 {total:,}건 이미지 분석 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")

        for col, label in [
            ("이미지유형",   "이미지 내용 유형 (Image Content Type)"),
            ("감정톤",       "감정적 어조 (Emotional Tone)"),
            ("후보표현방식", "후보자 표현 방식 (Candidate Presentation)"),
        ]:
            print(f"\n📈 {label} 분포:")
            for val, cnt in result_df[col].value_counts().items():
                print(f"   • {val}: {cnt:,}건 ({cnt/total*100:.1f}%)")

        sr = self.parsing_successes / total * 100 if total else 0
        print(f"\n📈 성공률: {self.parsing_successes}건 ({sr:.1f}%) / 실패: {self.parsing_failures}건")
        print(f"{'='*62}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 6. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════
#
# [모델 선택 가이드 — OpenAI Docs 기준]
#   gpt-5.4-mini  → 고볼륨 분류, agent workflow에 최적 (본 코드 기본값)
#   gpt-5.4       → 복잡한 추론, 지식 집약적 작업
#   gpt-5.4-nano  → 단순 고처리량 태스크 (비용 최소)
#
# [reasoning_effort 선택 가이드]
#   "none"   → 빠른 추출·분류·변환 태스크 ← 정치 이미지 분류 권장
#   "low"    → 레이턴시 민감 + 보통 복잡도
#   "medium" → 리서치, 합성, 멀티문서 검토
#   "high"   → 추론 집약적 에이전트
#   "xhigh"  → 최고 추론 깊이 (신규, eval로 효과 검증 후 사용)
#
# [image_detail 선택 가이드]
#   "high"     → 표준 이미지 시각 분석 ← 본 코드 기본값
#   "original" → 이미지 내 텍스트·배너·슬로건 포함 분석 시
#   "low"      → 속도 우선, 정밀도 낮음
#
# [temperature 주의]
#   reasoning_effort="none" 일 때만 사용 가능
#   다른 effort 수준에서 temperature 설정 시 API 오류!
# ═══════════════════════════════════════════════════════════════════════

MODEL            = "gpt-5.4-mini"              # ★ GPT-5.4-mini
API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")  # 환경변수 사용 권장. 키를 코드에 직접 적은 채 커밋하지 마세요.
INPUT_CSV_PATH   = "Input CSV file path here"   # image 컬럼(이미지 URL) 포함 CSV
OUTPUT_CSV_PATH  = "Output CSV file path here"

REASONING_EFFORT = "none"     # ★ 분류 태스크에는 "none" 권장
TEXT_VERBOSITY   = "medium"
IMAGE_DETAIL     = "high"     # "original" → 이미지 내 텍스트 포함 분석 시
TEMPERATURE      = None       # reasoning="none" 일 때만 유효 (예: 0.0~1.0)
# ※ GPT-4.1 버전은 temperature=0.0 고정이었습니다. 동일한 재현성이 필요하면 0.0으로 설정하세요(reasoning_effort="none"일 때만 유효).

DEBUG_MODE       = False

# 비용 추정용 단가 (USD / 1M tokens). OpenAI 공식 가격표 기준 gpt-5.4-mini 입력 $0.75, 출력 $4.50 (2026-11 확인). 가격이 바뀌면 수정하세요.
PRICE_INPUT_PER_1M  = 0.75
PRICE_OUTPUT_PER_1M = 4.50


async def main():
    try:
        df = pd.read_csv(INPUT_CSV_PATH, encoding="utf-8-sig")

        if "image" not in df.columns:
            print("❌ 'image' 컬럼이 필요합니다 (이미지 URL 문자열 포함)")
            return

        print(f"📥 총 {len(df):,}건 로드 완료")
        print(f"🤖 모델: {MODEL} | reasoning: {REASONING_EFFORT} | detail: {IMAGE_DETAIL}")
        print(f"🖼️  이미지 URL 예시: {df['image'].dropna().iloc[0]}")

        output_dir  = os.path.dirname(OUTPUT_CSV_PATH) or "."
        output_file = os.path.basename(OUTPUT_CSV_PATH)

        analyzer = PoliticalImageAnalyzer(
            api_key          = API_KEY,
            model            = MODEL,
            output_dir       = output_dir,
            reasoning_effort = REASONING_EFFORT,
            text_verbosity   = TEXT_VERBOSITY,
            image_detail     = IMAGE_DETAIL,
            temperature      = TEMPERATURE,
            debug            = DEBUG_MODE,
        )

        await analyzer.analyze_dataframe(df, output_file)

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except ValueError as e:
        print(f"❌ 설정 오류: {e}")
    except Exception as e:
        import traceback
        print(f"❌ 오류: {str(e)}")
        traceback.print_exc()


def run_analysis():
    """분석 실행 (Jupyter / VSCode 모두 호환)"""
    try:
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())
    except ImportError:
        asyncio.run(main())


if __name__ == "__main__":
    run_analysis()